In [1]:
# import libraries
import sqlalchemy as db
import pandas as pd

from dotenv import load_dotenv
import os

from data_warehouse.configuration.config import Config
from data_warehouse.src.db_connection import connect_to_mysql

load_dotenv("../configuration/.env")

True

In [2]:
# connect to MySQL database data warehouse
clientMysql= connect_to_mysql(Config.MYSQL_DATABASE_DW)

mysql://root:@localhost/dw_netflix


## Extract stage

The Excel file consists of two sheets: **Movies** and **Recommendations**. Here's the structure of each sheet:

|Movies Sheet|Recommendations Sheet |
| --- | --- |
name: Movie name | name: User name | 
|genre: Movie genre | movie: Movie title recommended by the user|
|performers: A list of actors who participated in the movie, formatted as 'name(role)' separated by '-'|genre: Movie genre topic of the post, indicating the genre for which users are seeking recommendations|
mentions: Number of times the movie has been mentioned. These mentions can be positive or negative|

In [3]:
# Load the Excel file and read data from each sheet
file_path = '../data/facebook_groups.xlsx'

df_movies = pd.read_excel(file_path,'movies')
df_recommendations = pd.read_excel(file_path,'recommendations')

In [4]:
# Check dataframes
print(df_movies.head())
print(df_recommendations.head())

            name      genre  \
0  Jurassic Park  Adventure   
1         Avatar     Sci-Fi   
2  Black Panther     Action   
3   The Avengers     Action   
4           Jaws     Horror   

                                          performers  mentions  
0  Jeff Goldblum(actor)-Sam Neill(actor)-Laura De...       203  
1  Sam Worthington(actor)-Zoe Saldana(actress)-Si...       153  
2  Chadwick Boseman(actor)-Michael B. Jordan(acto...       345  
3  Robert Downey Jr.(actor)-Chris Evans(actor)-Sc...       316  
4  Roy Scheider(actor)-Robert Shaw(actor)-Richard...       405  
              name                     movie   genre
0         John Doe                 Inception  Action
1      Alice Smith  The Shawshank Redemption   Drama
2  Michael Johnson           The Dark Knight  Action
3       Emma Brown              Pulp Fiction   Crime
4      Chris Davis                Fight Club   Drama


## Transform stage

In [5]:
df_dim_movies = df_movies[['name', 'genre']].rename(columns={'name': 'title'})
df_dim_movies = df_dim_movies.reset_index(names='movieID',allow_duplicates=False)
df_dim_movies['movieID'] = df_dim_movies['movieID'].astype(str)

Split performers from [name1(role1)-name2(role2)-...] into separate rows onto a new DataFrame

In [6]:
# function to extract performer details
def extract_performers(movies_df):
    data = []

    for index, row in movies_df.iterrows():
        movie_index_ref = index + 1
        movie_title = row['name']
        performers = row['performers']

        # Split the performers by commas and extract names and roles
        performers_list = performers.split('-')
        for performer in performers_list:
            name_role = performer.strip().split('(')
            performer_name = name_role[0].strip()
            performer_role = name_role[1].replace(')', '').strip() if len(name_role) > 1 else None

            # Append the data to the list
            data.append({
                'id': movie_index_ref,
                'movieTitle': movie_title,
                'performerName': performer_name,
                'performerRole': performer_role
            })

    # Create a new DataFrame from the extracted data
    performers_df = pd.DataFrame(data)
    return performers_df


In [7]:
df_performers = extract_performers(df_movies)
print(df_performers.head())

   id     movieTitle    performerName performerRole
0   1  Jurassic Park    Jeff Goldblum         actor
1   1  Jurassic Park        Sam Neill         actor
2   1  Jurassic Park       Laura Dern       actress
3   2         Avatar  Sam Worthington         actor
4   2         Avatar      Zoe Saldana       actress


Calculate score for each movie based on the numbre of recommendations

In [8]:
from numpy import NaN

def assign_score(quartile, median, value):
  if value >= quartile:
    return 100
  elif value >= median and value < quartile:
    return 80
  elif value>=1 and value < median:
    return 70
  else:
    return NaN

In [9]:
# Calculate score for each movie
total_recommendations_by_movie = df_recommendations['movie'].value_counts()
print(total_recommendations_by_movie.describe())
print(f"median: {total_recommendations_by_movie.median()}")

# Create a score dataframe merging df_movies and df_recommendations
df_movie_scores = pd.DataFrame(df_movies['name'].copy())
df_movie_scores['recommendations'] = df_movie_scores['name'].map(df_recommendations['movie'].value_counts())
df_movie_scores['score'] = df_movie_scores['recommendations'].apply(lambda x: assign_score(total_recommendations_by_movie.quantile(0.75), total_recommendations_by_movie.median(), x))

print(df_movie_scores.head(10))

count    52.000000
mean      3.519231
std       1.552864
min       1.000000
25%       3.000000
50%       3.000000
75%       4.000000
max       8.000000
Name: count, dtype: float64
median: 3.0
                    name  recommendations  score
0          Jurassic Park              NaN    NaN
1                 Avatar              2.0   70.0
2          Black Panther              NaN    NaN
3           The Avengers              2.0   70.0
4                   Jaws              NaN    NaN
5               Die Hard              NaN    NaN
6          The Godfather              5.0  100.0
7             La La Land              5.0  100.0
8          The Lion King              4.0  100.0
9  The Dark Knight Rises              3.0   80.0


In [10]:
# Prepare dimScore DataFrame
def generate_comment(score):
	movieComment = "excelent movie. One of my favourites. Highly recommended"
	if pd.isna(score):
		movieComment = ""
	elif score>=80 and score<90:
		movieComment = "very nice movie. I recommend it"
	elif score>=70 and score<80:
		movieComment = "good movie. Enjoyable"
	elif score>=60 and score<70:
		movieComment = "good to spend some time if there isn't anything else. Could be better"
	elif score>=50 and score<60:
		movieComment = "just another movie. Personally I didnt like it. Up to you to watch it"
	else:
		movieComment = "Awful. don't waste your time"
	return movieComment

df_movie_scores= df_movie_scores[['name', 'score']].rename(columns={
    'name': 'title',
    'calculated_score': 'peopleScore'
})
df_movie_scores['netflixScore'] = None
df_movie_scores['imdbScore'] = None
df_movie_scores['rottentomatoesScore'] = None
df_movie_scores['sensacineScore'] = None
df_movie_scores['peopleComment'] = df_movie_scores['score'].apply(lambda x: generate_comment(x))


generate interaction data, suppousing that a user who is recommending a movie will have a finishCount of 1, playCount of 1, backClickCount of 0, movieForwardCount of 0, and movieViewPercentage of 100

In [11]:
# Function to generate interaction data
def generate_interaction_data(row):
    finish_count = 1
    play_count = 1
    back_click_count = 0
    movie_forward_count = 0
    movie_view_percentage = 100.00

    return {
        'username':row['name'],
        'finishCount': finish_count,
        'backClickCount': back_click_count,
        'movieForwardCount': movie_forward_count,
        'playCount': play_count,
        'movieViewPercentage': movie_view_percentage
    }

# Apply the function to generate interaction data for each recommendation
interactions_data = df_recommendations.apply(generate_interaction_data, axis=1)
df_interactions = pd.DataFrame(list(interactions_data))
print(df_interactions.head())

          username  finishCount  backClickCount  movieForwardCount  playCount  \
0         John Doe            1               0                  0          1   
1      Alice Smith            1               0                  0          1   
2  Michael Johnson            1               0                  0          1   
3       Emma Brown            1               0                  0          1   
4      Chris Davis            1               0                  0          1   

   movieViewPercentage  
0                100.0  
1                100.0  
2                100.0  
3                100.0  
4                100.0  


## Load stage

In [12]:
# Load dimMovie data
df_dim_movies.to_sql('dimmovie', con=clientMysql, if_exists='append', index=False,index_label='movieID')

50

In [13]:
# Prepare dimUser DataFrame
df_dim_user = df_recommendations['name'].copy()
df_dim_user = pd.DataFrame(df_dim_user)
df_dim_user = df_dim_user.rename(columns={'name': 'username'}).drop_duplicates().reset_index(names='userID')
print(df_dim_user.head())

# Load dimUser data
df_dim_user.to_sql('dimuser', con=clientMysql, if_exists='append', index=False)

   userID         username
0       0         John Doe
1       1      Alice Smith
2       2  Michael Johnson
3       3       Emma Brown
4       4      Chris Davis


94

In [14]:
# prepare dimInteraction DataFrame
df_dim_interactions = df_interactions.copy()
df_dim_interactions.drop(columns=['username'], inplace=True)
df_dim_interactions = df_dim_interactions.reset_index(names='interactionID')

# Load dimInteraction data
df_dim_interactions.to_sql('diminteraction', con=clientMysql, if_exists='append', index=False)


183

In [15]:
#prepare dimScore DataFrame
df_dim_score = df_movie_scores.copy()
df_dim_score.drop(columns=['title'], inplace=True)
df_dim_score.rename(columns={'score': "peopleScore"}, inplace=True)
df_dim_score = df_dim_score.reset_index(names='scoreID')

# Load dimScore data
df_dim_score.to_sql('dimscore', con=clientMysql, if_exists='append', index=False, index_label='scoreID')


50

In [16]:
# prepare dimPerformer DataFrame
df_dim_performers = df_performers.copy()
df_dim_performers.drop(columns=['id','movieTitle'],inplace=True)
df_dim_performers.rename(columns={'performerName': 'name', 'performerRole': 'role'}, inplace=True)
df_dim_performers.reset_index(names='performerID', inplace=True)

# Load performers data
df_dim_performers.to_sql('dimperformer', con=clientMysql, if_exists='append', index=False)


127

merge the dimentions tables into one DataFrame to fill the factWatchs table

In [17]:
df_fact_watchs =df_recommendations.copy()

# Map users to their IDs
df_dim_user = pd.read_sql('SELECT userID,username FROM dimuser', con=clientMysql)
df_fact_watchs= df_fact_watchs.merge(df_dim_user, left_on='name', right_on='username').drop(columns=['name'])

# Map movies to their IDs
df_dim_movie= pd.read_sql('SELECT movieID,title FROM dimmovie', con=clientMysql)
df_fact_watchs = df_fact_watchs.merge(df_dim_movie, left_on='movie', right_on='title').drop(columns=['title'])

# Map performers to the movies they participate in
df_dim_performers= pd.read_sql('SELECT performerID FROM dimperformer', con=clientMysql)
df_dim_performers = df_dim_performers.merge(df_performers, left_on='performerID', right_on='id').drop(columns=['id'])
df_fact_watchs = df_fact_watchs.merge(df_dim_performers, left_on='movie', right_on='movieTitle').drop(columns=['movieTitle','performerName','performerRole'])

# Map scores to their movies
df_dim_score = df_movie_scores.copy()
df_dim_score = df_dim_score.reset_index(names='scoreID')
df_fact_watchs = df_fact_watchs.merge(df_dim_score, left_on='movie', right_on='title').drop(columns=['title'])

# Map interactions to their users
df_dim_interactions = df_interactions.copy().reset_index(names='interactionID')
df_fact_watchs = df_fact_watchs.merge(df_dim_interactions, on="username")

print(df_fact_watchs.columns)

Index(['movie', 'genre', 'userID', 'username', 'movieID', 'performerID',
       'scoreID', 'score', 'netflixScore', 'imdbScore', 'rottentomatoesScore',
       'sensacineScore', 'peopleComment', 'interactionID', 'finishCount',
       'backClickCount', 'movieForwardCount', 'playCount',
       'movieViewPercentage'],
      dtype='object')


In [18]:
# Prepare FactWatchs DataFrame
df_fact_watchs = df_fact_watchs[['userID', 'movieID','performerID','scoreID','interactionID']]
df_fact_watchs['movieID'] = df_fact_watchs['movieID'].astype(str)
df_fact_watchs.reset_index(names='id', inplace=True)

# Load FactWatchs data
df_fact_watchs.to_sql('factwatchs', con=clientMysql, if_exists='append', index=False)

print("Data loaded successfully")

Data loaded successfully
